# SCPC2026 제출 전 체크리스트

**제출 흐름 요약:**
1. 파일 경로 확인
2. screening_tasks 로드 → task ID 목록 확인
3. 각 task → 답안 객체 생성
4. 스키마 검증
5. 제출 CSV 생성

In [ ]:
import os
import sys
import subprocess
# subprocess.run([sys.executable, "-m", "pip", "install", "jsonschema", "-q"])

import csv
import json
import jsonschema

## Step 1. 파일 경로 확인
모든 경로 내 파일이 실제로 존재하는지 확인한다.

In [19]:
Root = "/Users/ksydata/SCPC2026/SCPC2026_Final_data"

Paths = {
    "screening_tasks" : f"{Root}/data/screening_tasks.jsonl",
    # 예선용/스크리닝 task 700개
    "dev_tasks" : f"{Root}/data/dev_tasks.jsonl",
    # 개발용/트레이닝 task
    "dev_answers" : f"{Root}/data/dev_answers.json", 
    # 개발용/트레이닝 task에 대한 정답 120개
    "schema" : f"{Root}/submission_schema.json",
    # 제출용 schema
    "sample" : f"{Root}/sample_submission.csv",
    # 제출전 참고용 sample
    "submit": "/Users/ksydata/SCPC2026/Sol/submission.csv",
}

'''
answer_template = {
    "focal_id" : "",               # ← task의 objects 중 핵심 대상 obj id
    "target" : "",                 # ← 최종 전달 대상 (수신처 이름)
    "control" : "proceed",         # ← proceed / amend / hold / ask 중 1개
    "content_scope" : {
        "mode": "summary",        # ← raw / summary / redacted / status_only / none
        "allowed_fields" : [],
        "excluded_fields" : [],
        "requires_user_confirmation" : False
    },
    "policy": {
        "risk_flags" : [],
        "violations" : [],
        "requires_confirmation" : False
    },
    "plan_events" : [],            # ← 최대 18개
    "user_response" : "",          # ← 선택 (사용자에게 보여줄 응답)
    "audit_tags" : []              # ← 선택
}
'''

for filename, path in Paths.items():
    exists = "✅" if os.path.exists(path) else "❌"
    print(f"{exists}  {filename}: {path}")

✅  screening_tasks: /Users/ksydata/SCPC2026/SCPC2026_Final_data/data/screening_tasks.jsonl
✅  dev_tasks: /Users/ksydata/SCPC2026/SCPC2026_Final_data/data/dev_tasks.jsonl
✅  dev_answers: /Users/ksydata/SCPC2026/SCPC2026_Final_data/data/dev_answers.json
✅  schema: /Users/ksydata/SCPC2026/SCPC2026_Final_data/submission_schema.json
✅  sample: /Users/ksydata/SCPC2026/SCPC2026_Final_data/sample_submission.csv
✅  submit: /Users/ksydata/SCPC2026/Sol/submission.csv


## Step 2. screening_tasks 로드 및 구조 확인
총 task 수와 첫 번째 task 구조를 확인한다.

In [21]:
tasks: list = []
# 예선용/스크리닝 task를 담을 리스트 생성
with open(Paths["screening_tasks"], "r") as file:
    # Paths 딕셔너리의 스크리닝 task 경로를 열기
    for line in file:
        line = line.strip()
        # 각 줄의 공백 제거

        if line:
            # task 리스트에 원소가 존재할 경우
            tasks.append(json.loads(line))
            #  각 줄을 읽어 json으로 변환하여 task 리스트에 추가

In [25]:
# 첫 번째 task의 구조 확인
print(f"총 task 수: {len(tasks)}")
print(f"\n[첫 번째 task 주요 필드]")
t = tasks[0]
print(f"  id            : {t['id']}")
print(f"  session_id    : {t['session_id']}")
print(f"  turn_index    : {t['turn_index']}")
print(f"  prompt        : {t['prompt'][ :200]}...")
print(f"  objects 수    : {len(t['device_state']['objects'])}")
print(f"  records 수    : {len(t['device_state']['records'])}")
print(f"  available_actions: {t['available_actions']}")

총 task 수: 700

[첫 번째 task 주요 필드]
  id            : final_screening_e6b1e73944de
  session_id    : final_screening_sess_0062
  turn_index    : 1
  prompt        : 원본, 익명 요약, 사진 중 현재 요청에 맞는 자료를 고르고, 보내도 되는 범위와 보류 여부를 판정해줘. 답을 정하기 전에 현재 record의 시간순서를 같이 검토해줘. 단, 요약 공유는 허용되지만 raw 문장과 위치, 숫자 값은 포함하지 않는다....
  objects 수    : 6
  records 수    : 8
  available_actions: ['read', 'verify', 'redact', 'summarize', 'dispatch', 'guard', 'clarify', 'update', 'schedule', 'toggle', 'pay']


## Step 3. 각 task → 답안 생성 (규칙 기반)

**분석된 패턴 (dev 120개 기준):**

| control | events 패턴 | 결정 신호 |
|---|---|---|
| `hold`   | read → guard   | consent=revoked, security_alert=impersonation, share_boundary=blocked+no_auth |
| `ask`    | read → clarify | authority_incomplete, target_changed_after_turn, dispatch_blocked+pending |
| `amend`  | read → redact → dispatch | share_boundary=redacted_external |
| `proceed`| read → verify → update | local_update, persistent_memory_write, internal_confirmed |

In [13]:
def get_rec(records, rec_type):
    for r in records:
        if isinstance(r, dict) and r.get("type") == rec_type:
            return r.get("value")
    return None

def has_rec(records, rec_type):
    return get_rec(records, rec_type) is not None

# 프롬프트에서 "로컬 전용 / 전송 취소" 키워드 감지
LOCAL_ONLY_KW = ["취소하고", "내부 업데이트", "기기 내부", "기기 안에서",
                 "바깥으로 보내지", "로컬 상태", "로컬 업데이트", "보내는 작업은 취소"]

def is_local_only_prompt(prompt: str) -> bool:
    return any(kw in prompt for kw in LOCAL_ONLY_KW)

def decide_control(records, prompt=""):
    consent        = get_rec(records, "consent")
    security       = get_rec(records, "security_alert")
    auth           = get_rec(records, "dispatch_authority_check")
    boundary       = get_rec(records, "share_boundary_update")
    ext_policy     = get_rec(records, "external_share_policy")
    route          = get_rec(records, "route_candidate_snapshot")
    binding_order  = get_rec(records, "route_binding_order")
    target_changed = has_rec(records, "target_changed_after_turn")
    duration_amb   = has_rec(records, "duration_ambiguous")

    # ── HOLD ────────────────────────────────────────────────────────────
    if consent == "revoked":
        return "hold"
    if security == "shared_thread_impersonation_suspected":
        return "hold"
    if (boundary == "dispatch_blocked_until_binding"
            and auth in (None, "authority_incomplete")
            and route != "external_candidates_present"):
        return "hold"

    # ── ASK ─────────────────────────────────────────────────────────────
    if auth == "authority_incomplete":
        return "ask"
    if target_changed:
        return "ask"
    if duration_amb:
        return "ask"
    if (boundary == "dispatch_blocked_until_binding"
            and auth == "user_binding_pending"
            and route == "external_candidates_present"):
        return "ask"

    # ── AMEND ───────────────────────────────────────────────────────────
    # 프롬프트가 "로컬 전용"을 명시하면 amend 대신 proceed
    local_only = is_local_only_prompt(prompt)

    if not local_only:
        if ext_policy in ("raw_quote_forbidden", "raw_sensitive_forbidden",
                          "summary_only_allowed", "doctor_note_forbidden"):
            return "amend"
    if boundary == "redacted_external_boundary":
        if not (auth == "internal_binding_confirmed"
                and binding_order == "authority_after_candidates"):
            return "amend"

    # ── PROCEED ─────────────────────────────────────────────────────────
    return "proceed"


EVENTS_MAP = {
    "hold":    [("read", "inspect_context"),   ("guard",    "precondition_invalidated")],
    "ask":     [("read", "inspect_context"),   ("clarify",  "clarification_required")],
    "amend":   [("read", "inspect_context"),   ("redact",   "sensitive_fields"),
                ("dispatch", "redacted")],
    "proceed": [("read", "inspect_context"),   ("verify",   "route_verified"),
                ("update", "local_update")],
}

def build_events(control, focal_id, target):
    events = []
    for verb, arg_val in EVENTS_MAP[control]:
        t = focal_id if verb in ("read", "redact", "guard") else target
        events.append({"verb": verb, "target": t, "args": {"purpose": arg_val}})
    return events

def get_focal_id(task):
    records = task["device_state"]["records"]
    objects = task["device_state"]["objects"]
    obj_by_ref = {o["attrs"].get("ref_code"): o["id"]
                  for o in objects if "ref_code" in o.get("attrs", {})}
    trace  = get_rec(records, "focal_resolution_trace")
    marker = get_rec(records, "focal_marker_refs")
    if trace and marker and isinstance(trace, dict) and isinstance(marker, dict):
        phase = trace.get("latest_phase")
        m2r   = marker.get("marker_to_ref", {})
        for mk, ref in m2r.items():
            if phase and phase in mk:
                return obj_by_ref.get(ref, objects[0]["id"])
        sel = trace.get("selected_marker") or trace.get("latest_marker")
        if sel and sel in m2r:
            return obj_by_ref.get(m2r[sel], objects[0]["id"])
    return objects[0]["id"] if objects else ""

def get_risk_flags(records, prompt=""):
    flags = set()
    boundary   = get_rec(records, "share_boundary_update")
    auth       = get_rec(records, "dispatch_authority_check")
    consent    = get_rec(records, "consent")
    security   = get_rec(records, "security_alert")
    policy     = get_rec(records, "session_share_policy")
    ext_policy = get_rec(records, "external_share_policy")
    if policy == "strict":                          flags.add("strict_share_policy")
    if boundary == "local_update_boundary":         flags.add("local_only")
    if boundary == "redacted_external_boundary":    flags.add("external_share")
    if boundary == "dispatch_blocked_until_binding":flags.update(["external_share", "precondition_invalidated"])
    if has_rec(records, "ambiguous_focal"):          flags.add("ambiguous_focal")
    if has_rec(records, "ambiguous_target"):         flags.add("target_ambiguity")
    if has_rec(records, "target_changed_after_turn"):flags.add("precondition_changed")
    if auth == "authority_incomplete":              flags.add("clarification_required")
    if consent == "revoked":                        flags.add("safety")
    if security:                                    flags.add("safety")
    if ext_policy:                                  flags.update(["sensitive_content", "minimal_disclosure"])
    if is_local_only_prompt(prompt):                flags.add("local_only")
    if not boundary or boundary == "local_update_boundary":
        flags.add("local_only")
    return sorted(flags)

def build_answer(task):
    records  = task["device_state"]["records"]
    objects  = task["device_state"]["objects"]
    prompt   = task.get("prompt", "")
    focal_id = get_focal_id(task)
    target   = get_rec(records, "resolved_target") or ""
    control  = decide_control(records, prompt)
    events   = build_events(control, focal_id, target)
    flags    = get_risk_flags(records, prompt)
    return {
        "focal_id": focal_id,
        "target":   target,
        "control":  control,
        "content_scope": {
            "mode": {"hold":"none","ask":"none","amend":"redacted","proceed":"summary"}.get(control,"summary"),
            "allowed_fields":             [],
            "excluded_fields":            [],
            "requires_user_confirmation": control in ("ask",),
        },
        "policy": {
            "risk_flags":            flags,
            "violations":            [],
            "requires_confirmation": control in ("ask",),
        },
        "plan_events":   events,
        "user_response": "",
        "audit_tags":    [],
    }

# ── 전체 screening 답안 재생성 ───────────────────────────────────────
answers = {task["id"]: build_answer(task) for task in tasks}

from collections import Counter
ctrl_dist = Counter(a["control"] for a in answers.values())
print(f"screening 답안 수: {len(answers)}")
print(f"control 분포: {dict(ctrl_dist)}")

screening 답안 수: 700
control 분포: {'proceed': 282, 'amend': 267, 'ask': 140, 'hold': 11}


## Step 3-1. dev 정답과 비교 (정확도 검증)
`dev_tasks`로 같은 로직을 돌려서 실제 정답과 얼마나 맞는지 확인한다.

In [14]:
dev_tasks = []
with open(paths["dev_tasks"]) as f:
    for line in f:
        line = line.strip()
        if line:
            dev_tasks.append(json.loads(line))

with open(paths["dev_answers"]) as f:
    dev_ref = json.load(f)
dev_ans = dev_ref["answers"]

# dev_tasks로 답안 생성
dev_pred = {t["id"]: build_answer(t) for t in dev_tasks}

# control 정확도
total = correct_ctrl = correct_events = 0
wrong_cases = []

for tid, ref_a in dev_ans.items():
    pred = dev_pred.get(tid)
    if not pred:
        continue
    total += 1

    ref_ctrl    = ref_a["control"]
    pred_ctrl   = pred["control"]
    ctrl_ok     = ref_ctrl == pred_ctrl

    ref_verbs   = [e["verb"] for e in ref_a.get("expected_events", [])]
    pred_verbs  = [e["verb"] for e in pred.get("plan_events", [])]
    events_ok   = ref_verbs == pred_verbs

    if ctrl_ok:
        correct_ctrl += 1
    if events_ok:
        correct_events += 1
    if not ctrl_ok:
        wrong_cases.append((tid, ref_ctrl, pred_ctrl))

print(f"총 dev task : {total}개")
print(f"control 정확도  : {correct_ctrl}/{total} = {correct_ctrl/total*100:.1f}%")
print(f"events  정확도  : {correct_events}/{total} = {correct_events/total*100:.1f}%")
print(f"\n[틀린 control 케이스 — 최대 10개]")
for tid, ref, pred in wrong_cases[:10]:
    print(f"  {tid[:30]}  정답={ref:8s}  예측={pred}")

총 dev task : 120개
control 정확도  : 65/120 = 54.2%
events  정확도  : 59/120 = 49.2%

[틀린 control 케이스 — 최대 10개]
  final_dev_e55d2c79fb78  정답=hold      예측=proceed
  final_dev_0ab2e0715082  정답=hold      예측=ask
  final_dev_8003c2e5b525  정답=amend     예측=proceed
  final_dev_88dbbfd07f1e  정답=proceed   예측=amend
  final_dev_6903fe98eb6a  정답=hold      예측=proceed
  final_dev_083ee82f08f6  정답=ask       예측=proceed
  final_dev_3541b9ea68b2  정답=amend     예측=proceed
  final_dev_891dd2e62a0a  정답=ask       예측=amend
  final_dev_0bd3e2e64880  정답=hold      예측=proceed
  final_dev_5eb14d5077e8  정답=ask       예측=proceed


## Step 4. 스키마 검증
생성한 답안(json)이 `submission_schema.json`을 통과하는지 확인한다.  
에러 메시지를 반환하면 어떤 필드에서 오류가 발생했는지 확인 후 step 3로 돌아가서 수정한다. 

In [33]:
with open(Paths["schema"]) as f:
    schema = json.load(f)
    # submission_schema 파일 불러오기

submission_json = {
    "schema": "scpc.final.answer.v1",
    "meta": {
        "harness_name" :      "my_harness",
        "uses_external_api" : False,
        "fixed_slm_policy" :  "local_fixed_slm_only",
        "model_id" :          "scpc-final-fixed-slm-local-facade",
        "temperature" :       0.0,
        "seed" :              42,
    },
    "answers": answers,
} # 제출용 json 스키마 파일

In [34]:
try:
    jsonschema.validate(instance=submission_json, schema = schema)
    print("✅ 스키마 검증 통과")
except jsonschema.ValidationError as e:
    print(f"❌ 검증 실패: {e.message}")
    print(f"   문제 위치: {list(e.path)}")

✅ 스키마 검증 통과


## Step 5. 제출 CSV 저장
검증 통과 후 `Sol/submission.csv`로 저장한다.  

In [27]:
os.makedirs(os.path.dirname(
    Paths["submit"]),
    exist_ok = True
)

with open(Paths["submit"], "w", newline = "", encoding = "utf-8") as f:
    # submission.csv 파일에 제출용 답안 저장(UTF-8 한글 인코딩)
    writer = csv.writer(f)
    writer.writerow(["submission"])
    # csv 파일의 첫 번째 행 헤더 작성
    writer.writerow([json.dumps(submission_json, ensure_ascii = False)])
    # csv 파일의 두 번째 행에 submission_json을 json 문자열로 변환하여 저장

In [30]:
print(f"✅ 저장 완료: {Paths['submit']}")
print(f"   답안 수  : {len(answers)}")

# 파일 크기 확인
size_KB = os.path.getsize(Paths["submit"]) / 1024
print(f"   파일 크기: {size_KB:.1f} KB")

✅ 저장 완료: /Users/ksydata/SCPC2026/Sol/submission.csv
   답안 수  : 700
   파일 크기: 512.0 KB
